# Merapi CaO_liq sensitivity test


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CACHE_DIR = PROJECT_ROOT / "paper" / ".cache" / "add_pre-2006_028"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "merapi_cao_plus4_all_cpx_liq"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

## Experimental design


In [ ]:
from paper.scripts.merapi_cao_plus4_all_cpx_liq import (
    DELTA_CAO_INPUT_WT,
    OXIDE_COLS,
    add_water_relation,
    build_plus4_liquid,
    calculate_all_models,
    load_accepted_pairs,
    summarize_results,
)

PAIRING_FILES = {
    2006: CACHE_DIR / "merapi_2006_pairing_kd028_err1e-4_n20000_endmembers.xlsx",
    2010: CACHE_DIR / "merapi_2010_pairing_kd028_err1e-4_n20000_endmembers.xlsx",
}

## Load accepted equilibrium pairs



In [ ]:
cpx_blocks = []
liq_blocks = []
meta_blocks = []

for eruption, pairing_path in PAIRING_FILES.items():
    cpx_block, liq_block, meta_block = load_accepted_pairs(eruption, pairing_path)
    cpx_blocks.append(cpx_block)
    liq_blocks.append(liq_block)
    meta_blocks.append(meta_block)

paired_cpx = pd.concat(cpx_blocks, ignore_index=True)
paired_liq_anhydrous = pd.concat(liq_blocks, ignore_index=True)
pair_meta = pd.concat(meta_blocks, ignore_index=True)

pair_meta.groupby("eruption").size().rename("accepted_pairs")

## Build baseline and CaO +4 wt.% liquids


In [ ]:
baseline_liq = add_water_relation(paired_liq_anhydrous)
adjusted_liq = build_plus4_liquid(paired_liq_anhydrous)

composition_audit = pair_meta.copy()
composition_audit["CaO_original_wt%"] = baseline_liq["CaO"]
composition_audit["CaO_target_pre_norm_wt%"] = baseline_liq["CaO"] + DELTA_CAO_INPUT_WT
composition_audit["CaO_plus4_normalized_wt%"] = adjusted_liq["CaO"]
composition_audit["actual_delta_CaO_normalized_wt%"] = (
    adjusted_liq["CaO"] - baseline_liq["CaO"]
)
composition_audit["normalized_oxide_sum_wt%"] = adjusted_liq[OXIDE_COLS].sum(axis=1)

composition_audit.groupby("eruption").agg(
    n=("pair_id", "size"),
    median_CaO_original_wt=("CaO_original_wt%", "median"),
    median_CaO_plus4_wt=("CaO_plus4_normalized_wt%", "median"),
    median_actual_delta_CaO_wt=("actual_delta_CaO_normalized_wt%", "median"),
)

## Calculate all cpx–liquid models

Baseline and adjusted liquids are calculated with the same random seed for each model.

In [ ]:
results, model_manifest = calculate_all_models(
    paired_cpx,
    baseline_liq,
    adjusted_liq,
    pair_meta,
)
summary = summarize_results(results)
summary

## Save results

In [ ]:
results_csv = OUTPUT_DIR / "merapi_cao_plus4_all_cpx_liq_results.csv"
summary_csv = OUTPUT_DIR / "merapi_cao_plus4_all_cpx_liq_summary.csv"
composition_csv = OUTPUT_DIR / "merapi_cao_plus4_composition_audit.csv"
models_csv = OUTPUT_DIR / "merapi_cao_plus4_model_manifest.csv"

results.to_csv(results_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
composition_audit.to_csv(composition_csv, index=False, encoding="utf-8-sig")
model_manifest.to_csv(models_csv, index=False, encoding="utf-8-sig")

results_csv, summary_csv, composition_csv, models_csv